In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append('../../scripts')

In [ ]:
import numpy as np
import scanpy as sc
import os
DATA_ROOT = os.environ.get("DATA_ROOT", ".")
import matplotlib.pyplot as plt
import decoupler as dc
import scipy.sparse as sp
import pandas as pd

from train_loo import preprocess_crc

In [ ]:
adata = sc.read_h5ad(os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_210.h5ad"))
adata = preprocess_crc(adata, n_top_genes=2000, n_neighbors=200, labels_key='coarse_type', domains_key='typ_clean')

In [ ]:
mask_crc_epi = (
    (adata.obs["coarse_type"] == "Epithelial") &
    (adata.obs["typ_clean"] == "CRC")
)

# Create a copy for visualization
adata.obs["mask1"] = np.where(mask_crc_epi, "removed", "kept")

In [ ]:
sc.pl.spatial(
    adata,
    color="mask1",
    palette={"removed": "lightgrey", "kept": "blue"},
    spot_size=100,
)

In [ ]:
A = adata.obsp["spatial_connectivities"]  # sparse matrix
mask_crc_epi = mask_crc_epi.values  # convert to numpy

# Multiply adjacency with mask to find neighbors
neighbor_mask = A.dot(mask_crc_epi.astype(int)) > 0

In [ ]:
mask_total = mask_crc_epi | neighbor_mask

In [ ]:
adata.obs["mask2"] = "kept"
adata.obs.loc[mask_crc_epi, "mask2"] = "CRC_epithelial"
adata.obs.loc[neighbor_mask, "mask2"] = "neighbor_of_CRC_epi"

sc.pl.spatial(
    adata,
    color="mask2",
    palette={
        "kept": "blue",
        "CRC_epithelial": "red",
        "neighbor_of_CRC_epi": "orange"
    },
    spot_size=100,
)

# Quantitative

In [ ]:
is_crc = adata.obs["typ_clean"] == "CRC"
is_celltype = "Epithelial"

mask_crc_epi = (
    (adata.obs["coarse_type"] == is_celltype) &
    (adata.obs["typ_clean"] == "CRC")
)

A = adata.obsp["spatial_connectivities"]

# optional: remove self-loops
A = A.copy()
A.setdiag(0)

neighbor_mask = A.dot(mask_crc_epi.astype(int).values) > 0

mask_total = mask_crc_epi.values | neighbor_mask

In [ ]:
crc_remaining_after_epi = np.sum(is_crc & ~mask_crc_epi)

crc_original = np.sum(is_crc)
crc_removed_epi = np.sum(is_crc & mask_crc_epi)

In [ ]:
crc_remaining_after_total = np.sum(is_crc.values & ~mask_total)

crc_removed_total = np.sum(is_crc.values & mask_total)

In [ ]:
frac_remaining_epi = crc_remaining_after_epi / crc_original
frac_remaining_total = crc_remaining_after_total / crc_original

print(f"Fraction remaining (no {is_celltype}): {frac_remaining_epi:.3f}")
print(f"Fraction remaining (no {is_celltype} + neighbors): {frac_remaining_total:.3f}")

In [ ]:
frac_remaining_epi = crc_remaining_after_epi / crc_original
frac_remaining_total = crc_remaining_after_total / crc_original

print(f"Fraction remaining (no {is_celltype}): {frac_remaining_epi:.3f}")
print(f"Fraction remaining (no {is_celltype} + neighbors): {frac_remaining_total:.3f}")